# MMF Prediction-Interval Calibration
Auto-generated by mmf-agent. Calibrates the prediction intervals of the latest run of each model, from the tables the forecast jobs wrote (`evaluation_output`, `scoring_output`) and the training data. It does not use the models or a GPU: run it on CPU (a classic CPU cluster or serverless CPU), after the forecast jobs.

It writes, appending, `{use_case}_evaluation_output_calibration`, `{use_case}_evaluation_output_interval_metrics` and `{use_case}_scoring_output_intervals`. The forecast tables are not modified, so this notebook can be re-run with other settings without re-running the models.

In [ ]:
%pip install "mmf_sa @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@main" --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
train_table = "{train_table}"
freq = "{freq}"
group_id = "{group_id}"
date_col = "{date_col}"
target = "{target}"

# Calibrated interval levels (Skill 4 Step 1d), and the interval columns of scoring_output_intervals.
levels = {calibration_levels}
output_format = "{output_format}"

# Pooling (Skill 4 Step 3f): "marginal" pools all series (default); "group" pools per group of group_col, a column
# of the training data (e.g. from Skill 2 profiling), or "auto" for MMF's ABC x XYZ classes (for intermittent demand only).
pool = "{pool}"
group_col = {group_col}

# Also calibrate the non-forecastable pipeline's tables ({use_case}_nf_*), with separate_job (Skill 4 Step 0a).
calibrate_non_forecastable = {calibrate_non_forecastable}
nf_train_table = use_case + "_train_data_non_forecastable"

In [ ]:
import json
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import calibrate_intervals

prefix = catalog + "." + schema + "."


def calibrate(tables_use_case, train):
    return calibrate_intervals(
        spark=spark,
        train_data=prefix + train,
        evaluation_output=prefix + tables_use_case + "_evaluation_output",
        scoring_output=prefix + tables_use_case + "_scoring_output",
        group_id=group_id,
        date_col=date_col,
        target=target,
        freq=freq,
        levels=levels,
        output_format=output_format,
        pool=pool,
        group_col=group_col,
        use_case=tables_use_case,
    )


results = {use_case: calibrate(use_case, train_table)}
if calibrate_non_forecastable:
    results[use_case + "_nf"] = calibrate(use_case + "_nf", nf_train_table)

In [ ]:
# Fail the job if a model could not be calibrated, so that the orchestrating job reports it.
failed = {uc + ": " + model: summary["error"] for uc, out in results.items()
          for model, summary in out["models"].items() if "error" in summary}
for uc, out in results.items():
    for model, summary in out["models"].items():
        print(uc, model, summary.get("methods"), summary.get("coverage"))
assert results[use_case]["models"], "No model was calibrated: " + use_case + "_evaluation_output has no backtest rows."
assert not failed, "Calibration failed for: " + json.dumps(failed)
dbutils.notebook.exit(json.dumps(results, default=str))